In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
df = pd.read_csv("data1C/model_option_A.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["id", "date"]).reset_index(drop=True)

if "mood_next_day" not in df.columns:
    df["mood_next_day"] = df.groupby("id")["mood"].shift(-1)

# =========================
# 2. Zero fill app cols (safe before split)
# =========================
zero_fill_cols = [
    "screen", "call", "sms",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
    "appCat.social", "appCat.travel", "appCat.unknown",
    "appCat.utilities", "appCat.weather"
]
for col in zero_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# =========================
# 3. Split per patient temporally
# =========================
train_data, test_data = [], []

for patient_id, group in df.groupby("id"):
    group = group.sort_values("date").copy()
    if len(group) < 10:
        continue
    split = int(0.8 * len(group))
    if split == 0 or split == len(group):
        continue
    train_data.append(group.iloc[:split])
    test_data.append(group.iloc[split:])

train_df = pd.concat(train_data).reset_index(drop=True)
test_df  = pd.concat(test_data).reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

# =========================
# 4. Interpolate + engineer features (separately on train and test)
# =========================
self_report_cols = ["mood", "activity", "circumplex.arousal", "circumplex.valence"]

def engineer_features(df):
    df = df.copy()

    # Interpolate self-report cols within each patient
    for col in [c for c in self_report_cols if c in df.columns]:
        df[col] = df.groupby("id")[col].transform(
            lambda x: x.interpolate(method="linear", limit_direction="both")
        )

    # Lag-1
    for col in ["mood", "activity", "screen"]:
        if col in df.columns:
            df[f"{col}_lag1"] = df.groupby("id")[col].shift(1)

    # 3-day mean
    mean_3d_features = [
        "mood", "activity", "screen", "call", "sms",
        "appCat.builtin", "appCat.communication", "appCat.entertainment",
        "appCat.finance", "appCat.game", "appCat.office", "appCat.other",
        "appCat.social", "appCat.travel", "appCat.unknown",
        "appCat.utilities", "appCat.weather",
        "circumplex.arousal", "circumplex.valence"
    ]
    for col in [c for c in mean_3d_features if c in df.columns]:
        shifted = df.groupby("id")[col].shift(1)
        df[f"{col}_mean_3d_hist"] = (
            shifted.groupby(df["id"])
            .rolling(3, min_periods=1).mean()
            .reset_index(level=0, drop=True)
        )

    # 3-day std
    for col in ["mood", "activity", "screen"]:
        if col in df.columns:
            shifted = df.groupby("id")[col].shift(1)
            df[f"{col}_std_3d_hist"] = (
                shifted.groupby(df["id"])
                .rolling(3, min_periods=2).std()
                .reset_index(level=0, drop=True)
            )
            df[f"{col}_std_3d_hist"] = df[f"{col}_std_3d_hist"].fillna(0)

    # Change features
    for col in ["mood", "activity", "screen"]:
        if f"{col}_lag1" in df.columns:
            df[f"{col}_change_today_vs_yesterday"] = df[col] - df[f"{col}_lag1"]

    # Calendar
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    # Drop rows without enough history
    required = ["mood_lag1", "activity_lag1", "screen_lag1", "mood_next_day"]
    required = [c for c in required if c in df.columns]
    df = df.dropna(subset=required).copy()

    return df

train_df = engineer_features(train_df)
test_df  = engineer_features(test_df)

print("Train shape after engineering:", train_df.shape)
print("Test shape after engineering:", test_df.shape)

# =========================
# 5. Save
# =========================
train_df.to_csv("data1C/train_data_1C.csv", index=False)
test_df.to_csv("data1C/test_data_1C.csv", index=False)

print("Saved: data1C/train_data_1C.csv")
print("Saved: data1C/test_data_1C.csv")